# Implicit Decision Gate: a verified long-running agent walkthrough

**Actors:** job owner, gate orchestrator, coding agent, PostgreSQL verifier, evidence reviewer, and deterministic gate.

## Premise

An engineering brief asks for 30-day expiration on 1Password item-sharing links. It specifies what new links must do but says nothing about existing links. A migration necessarily produces observable behavior for existing records; this demo models two acceptable outcomes:

- `PRESERVE_EXISTING`: existing links remain non-expiring; new links expire after 30 days.
- `EXPIRE_EXISTING`: existing links receive an expiration approximately 30 days after migration; new links expire after 30 days.

Either policy may be legitimate, but that product choice belongs to the job owner, not the coding agent.

## Motivation

This notebook follows one job across a durable agent loop: pin the human-owned inputs, run a fresh coding process, inspect its SQL, verify actual database behavior, ask a narrow evidence question against the brief, pause when intent is missing, record a typed owner decision, resume with a fresh coding process, and deterministically verify the result. Evidence comes from PostgreSQL, not from the coding agent's account of its own work.

The notebook drives the same public `idg` CLI as the terminal demo and then opens its persisted artifacts. It is an observability layer, not a second implementation of the orchestrator. `start` performs coding, probing, and evidence review as one operation; the cells after it unpack those persisted stages in causal order.

Execute the state-changing `start`, `answer`, and `resume` cells once, from top to bottom. They are intentionally non-idempotent. To repeat the walkthrough, rerun from `start` and continue with the new run identifier.

This checked-in copy includes outputs from one genuine live execution. Rerunning it creates a new run with new identifiers, timestamps, SQL, and model classifications; the first migration may choose either supported rollout policy.

> The current implementation does not compile the brief into a general semantic contract. It asks one domain-specific question about one behavior observed at runtime. This notebook makes that boundary visible.


## 1. Establish the notebook boundary

**Actor:** notebook operator.

This cell locates the repository and defines small display helpers. All state-changing work still goes through `uv run idg`; the helpers only execute commands and read the files that the application persists.


In [1]:
from __future__ import annotations

import copy
import difflib
import hashlib
import json
import os
import shlex
import shutil
import stat
import subprocess
from pathlib import Path
from typing import Any


def find_repo_root(start: Path) -> Path:
    for candidate in (start, *start.parents):
        if (candidate / "pyproject.toml").is_file() and (
            candidate / "examples/share-link-expiration/brief.md"
        ).is_file():
            return candidate.resolve()
    raise RuntimeError("Open this notebook from inside the implicit-decision-gate repository")


REPO_ROOT = find_repo_root(Path.cwd())
os.chdir(REPO_ROOT)
COMMAND_ENV = os.environ.copy()
COMMAND_ENV.pop("VIRTUAL_ENV", None)


def run_command(
    arguments: list[str],
    *,
    show_output: bool = True,
    check: bool = True,
) -> subprocess.CompletedProcess[str]:
    completed = subprocess.run(
        arguments,
        cwd=REPO_ROOT,
        capture_output=True,
        text=True,
        check=False,
        env=COMMAND_ENV,
    )
    if show_output:
        print(f"$ {shlex.join(arguments)}")
        if completed.stdout:
            print(completed.stdout.rstrip())
        if completed.stderr:
            print(completed.stderr.rstrip())
    if check and completed.returncode != 0:
        raise RuntimeError(
            f"Command exited with status {completed.returncode}: {shlex.join(arguments)}"
        )
    return completed


def run_idg(*arguments: str) -> dict[str, Any]:
    completed = run_command(["uv", "run", "idg", *arguments], check=False)
    try:
        payload = json.loads(completed.stdout)
    except json.JSONDecodeError as error:
        raise RuntimeError("idg did not return its expected JSON summary") from error
    if completed.returncode != 0:
        print(
            f"idg exited with status {completed.returncode}; "
            "its persisted summary remains inspectable"
        )
    return payload


def load_run(run_id: str) -> dict[str, Any]:
    path = REPO_ROOT / ".idg" / "runs" / run_id / "run.json"
    return json.loads(path.read_text(encoding="utf-8"))


print(f"Repository: {REPO_ROOT}")

Repository: /home/user/projects/implicit-decision-gate


## 2. Check the execution environment

**Actor:** notebook operator.

The live path needs Git and `uv`, an installed and authenticated Codex CLI, and PostgreSQL 17. Docker is required only when it supplies that disposable PostgreSQL instance; an external admin DSN can be provided with `IDG_POSTGRES_ADMIN_DSN`.


In [2]:
EXTERNAL_POSTGRES = bool(os.environ.get("IDG_POSTGRES_ADMIN_DSN"))
required_tools = ["git", "uv", "codex"]
if not EXTERNAL_POSTGRES:
    required_tools.append("docker")

tool_paths = {name: shutil.which(name) for name in required_tools}
print(json.dumps(tool_paths, indent=2))
missing_tools = [name for name, path in tool_paths.items() if path is None]
if missing_tools:
    raise RuntimeError(f"Missing required tools: {', '.join(missing_tools)}")

run_command(["git", "rev-parse", "HEAD"])
run_command(["uv", "--version"])
run_command(["codex", "--version"])
if EXTERNAL_POSTGRES:
    print("PostgreSQL source: IDG_POSTGRES_ADMIN_DSN is configured; its value is hidden.")
else:
    run_command(["docker", "compose", "version"])

{
  "git": "/usr/bin/git",
  "uv": "/home/user/.local/bin/uv",
  "codex": "/home/user/.local/bin/codex",
  "docker": "/usr/bin/docker"
}
$ git rev-parse HEAD
7703bce2b4afc78a98765346decf7953b032d91b
$ uv --version
uv 0.11.25 (x86_64-unknown-linux-gnu)
$ codex --version
codex-cli 0.149.0
$ docker compose version
Docker Compose version v5.3.0


## 3. Start the disposable verifier

**Actors:** notebook operator and PostgreSQL verifier.

When no external DSN is configured, Compose starts PostgreSQL 17. PostgreSQL is part of the trust argument: it executes the real DDL and exposes default, backfill, nullability, and rollback behavior that SQL text inspection alone cannot establish.


In [3]:
if EXTERNAL_POSTGRES:
    print("Using the configured disposable PostgreSQL instance.")
else:
    run_command(["docker", "compose", "up", "-d", "--wait"])

$ docker compose up -d --wait
 Network implicit-decision-gate_default Creating 
 Network implicit-decision-gate_default Created 
 Container implicit-decision-gate-postgres-1 Creating 
 Container implicit-decision-gate-postgres-1 Created 
 Container implicit-decision-gate-postgres-1 Starting 
 Container implicit-decision-gate-postgres-1 Started 
 Container implicit-decision-gate-postgres-1 Waiting 
 Container implicit-decision-gate-postgres-1 Healthy


## 4. Inspect the authoritative inputs

**Actors:** job owner for the brief, service repository for the baseline schema, and Git for provenance.

The brief and schema below are read from the current commit, not from uncommitted working-tree files. The seeded `existing-fixture` row is what makes the missing rollout policy observable. At this point the brief is exact source text; there is no precomputed semantic representation.


In [4]:
HEAD_BEFORE_START = run_command(["git", "rev-parse", "HEAD"], show_output=False).stdout.strip()
BRIEF_PATH = "examples/share-link-expiration/brief.md"
SCHEMA_PATH = "examples/share-link-expiration/schema.sql"
brief_at_head = run_command(
    ["git", "show", f"{HEAD_BEFORE_START}:{BRIEF_PATH}"], show_output=False
).stdout
schema_at_head = run_command(
    ["git", "show", f"{HEAD_BEFORE_START}:{SCHEMA_PATH}"], show_output=False
).stdout

print(f"Pinned candidate commit: {HEAD_BEFORE_START}")
print("\nAuthoritative brief:\n")
print(brief_at_head)
print("Baseline schema and fixture:\n")
print(schema_at_head)

Pinned candidate commit: 7703bce2b4afc78a98765346decf7953b032d91b

Authoritative brief:

Add 30-day expiration support to item-sharing links.

Store expiration in `public.share_links.expires_at` as a nullable timestamp with time
zone. New item-sharing links must expire 30 days after creation.

Baseline schema and fixture:

CREATE TABLE public.share_links (
    id bigint GENERATED ALWAYS AS IDENTITY PRIMARY KEY,
    token text NOT NULL UNIQUE,
    created_at timestamp with time zone NOT NULL DEFAULT CURRENT_TIMESTAMP
);

INSERT INTO public.share_links (token) VALUES ('existing-fixture');



## 5. Start one durable run

**Actor:** gate orchestrator.

`start` pins the current commit, creates a clean detached worktree, invokes a fresh Codex process for SQL, probes that SQL in PostgreSQL, and invokes a separate fresh Codex process for the evidence question. It returns only after those stages are persisted. The following cells inspect the captured start snapshot in their causal order. Because the model calls are live, an unexpected terminal state is displayed and stops this walkthrough instead of being forced into the expected story; rerun this cell to create another independent run.


In [5]:
start_summary = run_idg("start")
RUN_ID = str(start_summary["run_id"])
RUN_DIR = REPO_ROOT / ".idg" / "runs" / RUN_ID
START_SNAPSHOT = copy.deepcopy(load_run(RUN_ID))

if START_SNAPSHOT["state"] != "AWAITING_OWNER":
    raise RuntimeError(
        "This live run did not enter AWAITING_OWNER. Inspect the summary above; "
        "model output is nondeterministic, and the walkthrough must not pretend it paused."
    )
print(f"\nDurable run directory: {RUN_DIR}")

$ uv run idg start
{
  "run_id": "696f73f7991f46768159e7dcf2e05ad5",
  "state": "AWAITING_OWNER",
  "observed_option": "EXPIRE_EXISTING",
  "classification": "NOT_EVIDENCED",
  "decision_request": {
    "id": "existing_item_sharing_link_rollout",
    "question": "What should happen to existing item-sharing links?",
    "reason": "The gate could not establish from the brief whether the 30-day expiration should apply to existing item-sharing links.",
    "observed": {
      "option": "EXPIRE_EXISTING",
      "behavior": "Existing item-sharing links receive an expiration approximately 30 days from migration; new links default to approximately 30 days after creation; expires_at remains nullable."
    },
    "options": [
      {
        "option": "PRESERVE_EXISTING",
        "behavior": "Existing item-sharing links remain non-expiring with NULL; new links default to approximately 30 days after creation; expires_at remains nullable.",
        "command": "uv run idg answer 696f73f7991f4676815

## 6. Verify the pinned run envelope

**Actors:** gate orchestrator and Git.

The run stores the commit and the authoritative brief verbatim. These checks establish the provenance of the pinned inputs; the next cell shows the exact rendered prompt that the application supplied to Codex.


In [6]:
pinned_commit = str(START_SNAPSHOT["base_commit"])
pinned_brief = run_command(
    ["git", "show", f"{pinned_commit}:{BRIEF_PATH}"], show_output=False
).stdout
pinned_schema = run_command(
    ["git", "show", f"{pinned_commit}:{SCHEMA_PATH}"], show_output=False
).stdout

envelope = {
    "run_id": START_SNAPSHOT["run_id"],
    "state": START_SNAPSHOT["state"],
    "base_commit": pinned_commit,
    "created_at": START_SNAPSHOT["created_at"],
    "updated_at": START_SNAPSHOT["updated_at"],
    "original_brief": START_SNAPSHOT["original_brief"],
    "brief_matches_pinned_commit": START_SNAPSHOT["original_brief"] == pinned_brief,
    "schema_matches_pre_start_commit": pinned_schema == schema_at_head,
}
print(json.dumps(envelope, indent=2))
attempt_one_prompt = str(START_SNAPSHOT["attempts"][0]["coding_prompt"])
assert pinned_commit == HEAD_BEFORE_START
assert pinned_brief in attempt_one_prompt
assert pinned_schema in attempt_one_prompt
assert envelope["brief_matches_pinned_commit"]
assert envelope["schema_matches_pre_start_commit"]

{
  "run_id": "696f73f7991f46768159e7dcf2e05ad5",
  "state": "AWAITING_OWNER",
  "base_commit": "7703bce2b4afc78a98765346decf7953b032d91b",
  "created_at": "2026-08-23T14:33:37.526080Z",
  "updated_at": "2026-08-23T14:34:29.495220Z",
  "original_brief": "Add 30-day expiration support to item-sharing links.\n\nStore expiration in `public.share_links.expires_at` as a nullable timestamp with time\nzone. New item-sharing links must expire 30 days after creation.\n",
  "brief_matches_pinned_commit": true,
  "schema_matches_pre_start_commit": true
}


## 7. Inspect the first coding request

**Actor:** gate prompt renderer.

This is the exact project-controlled prompt persisted for attempt one. The coding agent receives the original brief, baseline schema, and a narrow structured-output contract. It receives no typed answer for the omitted existing-link policy.


In [7]:
from implicit_decision_gate.codex_client import CODING_SCHEMA

attempt_one = START_SNAPSHOT["attempts"][0]
print("Persisted coding prompt:\n")
print(attempt_one["coding_prompt"])
print("\nCodex structured-output schema:\n")
print(json.dumps(CODING_SCHEMA, indent=2))

Persisted coding prompt:

You create exactly one PostgreSQL migration.
Use only the supplied brief and baseline schema. Do not inspect or edit repository files.
Return the complete migration as structured SQL, without transaction-control statements.
The expires_at column must be a nullable timestamp with time zone. New rows must default
to approximately 30 days from creation.

Original brief:
Add 30-day expiration support to item-sharing links.

Store expiration in `public.share_links.expires_at` as a nullable timestamp with time
zone. New item-sharing links must expire 30 days after creation.


Baseline schema:
CREATE TABLE public.share_links (
    id bigint GENERATED ALWAYS AS IDENTITY PRIMARY KEY,
    token text NOT NULL UNIQUE,
    created_at timestamp with time zone NOT NULL DEFAULT CURRENT_TIMESTAMP
);

INSERT INTO public.share_links (token) VALUES ('existing-fixture');


Codex structured-output schema:

{
  "type": "object",
  "properties": {
    "sql": {
      "type": "string"


## 8. Inspect the first proposed migration

**Actors:** coding agent for the proposal, run store for the immutable artifact, and Git worktree manager for isolation.

The migration has three useful representations in this system: SQL as the proposed mechanism, a hash-identified immutable artifact, and normalized behavior produced by PostgreSQL. This cell shows the first two and independently recomputes the artifact digest.


In [8]:
artifact_one_path = RUN_DIR / "attempt-1.sql"
migration_one = artifact_one_path.read_text(encoding="utf-8")
computed_digest_one = hashlib.sha256(migration_one.encode()).hexdigest()
worktree_one = Path(str(attempt_one["worktree_path"]))
worktree_one_head = run_command(
    ["git", "-C", str(worktree_one), "rev-parse", "HEAD"],
    show_output=False,
).stdout.strip()
worktree_one_status = run_command(
    ["git", "-C", str(worktree_one), "status", "--short"],
    show_output=False,
).stdout.rstrip()

print("Attempt-one SQL artifact:\n")
print(migration_one)
artifact_one = {
    "path": str(artifact_one_path),
    "stored_digest": attempt_one["migration_digest"],
    "computed_digest": computed_digest_one,
    "digest_matches": attempt_one["migration_digest"] == computed_digest_one,
    "file_mode": oct(stat.S_IMODE(artifact_one_path.stat().st_mode)),
    "worktree_path": str(worktree_one),
    "clean_start_verified_before_write": attempt_one["clean_start_verified"],
    "worktree_head": worktree_one_head,
    "worktree_matches_base_commit": worktree_one_head == pinned_commit,
    "worktree_status_after_write": worktree_one_status,
}
print(json.dumps(artifact_one, indent=2))
assert artifact_one["digest_matches"]
assert artifact_one["worktree_matches_base_commit"]

Attempt-one SQL artifact:

ALTER TABLE public.share_links
    ADD COLUMN expires_at timestamp with time zone
    DEFAULT (CURRENT_TIMESTAMP + INTERVAL '30 days');

{
  "path": "/home/user/projects/implicit-decision-gate/.idg/runs/696f73f7991f46768159e7dcf2e05ad5/attempt-1.sql",
  "stored_digest": "78122b760223f63aeed3c26e56802fcc797136feac262d233196872bef232bf3",
  "computed_digest": "78122b760223f63aeed3c26e56802fcc797136feac262d233196872bef232bf3",
  "digest_matches": true,
  "file_mode": "0o444",
  "worktree_path": "/home/user/projects/.implicit-decision-gate-idg-worktrees/696f73f7991f46768159e7dcf2e05ad5/attempt-1",
  "clean_start_verified_before_write": true,
  "worktree_head": "7703bce2b4afc78a98765346decf7953b032d91b",
  "worktree_matches_base_commit": true,
  "worktree_status_after_write": "?? examples/share-link-expiration/migrations/idg-696f73f7991f46768159e7dcf2e05ad5-attempt-1.sql"
}


## 9. Inspect normalized runtime evidence

**Actor:** PostgreSQL verifier.

The verifier applied the baseline schema and migration in a disposable database, inspected the column and two representative rows, then rolled the transaction back. A modeled policy requires the exact PostgreSQL type, a nullable column, a non-null default, and one immediate insert whose expiration is within 10 seconds of migration time plus 30 days. The existing-row label then distinguishes the two accepted policies; anything else is `UNMODELED` and fails before evidence review.

The raw timestamps exist only during the probe; `run.json` retains the normalized facts below. This bounded check does not prove how an arbitrary future link relates to its own `created_at`. The `ProbeResult` is the migration's behavioral representation for this demo contract.


In [9]:
probe_one = attempt_one["probe_result"]
print(json.dumps(probe_one, indent=2))

{
  "data_type": "timestamp with time zone",
  "nullable": true,
  "column_default": "(CURRENT_TIMESTAMP + '30 days'::interval)",
  "insert_without_value": "approximately_now_plus_30_days",
  "existing_row": "approximately_migration_time_plus_30_days",
  "rollout_option": "EXPIRE_EXISTING",
  "rollback_verified": true
}


## 10. Inspect the narrow evidence review

**Actor:** evidence reviewer, running in a separate fresh Codex process.

The reviewer does not analyze every meaning in the brief. It receives the exact brief plus one observed rollout hypothesis and asks whether that behavior is explicitly supported. Its structured-output schema requires a `classification` from `SUPPORTED`, `CONTRADICTED`, `NOT_EVIDENCED`, or `UNCERTAIN`, plus an `evidence_quote` string, with no additional fields. The adapter stores an empty quote as `null`.

The persisted AI-derived result is only `{classification, evidence_quote}`; supported or contradicted quotes are additionally checked as literal substrings of the brief.


In [10]:
print("Persisted reviewer prompt:\n")
print(START_SNAPSHOT["reviewer_prompt"])
print("\nValidated reviewer result:\n")
print(json.dumps(START_SNAPSHOT["reviewer_result"], indent=2))

# This is an inspection view over persisted fields, not another stored model.
brief_after_review_view = {
    "authoritative_brief": START_SNAPSHOT["original_brief"],
    "observed_hypothesis": START_SNAPSHOT["decision"]["observed"],
    "reviewer_result": START_SNAPSHOT["reviewer_result"],
}
print("\nWhat exists after review:\n")
print(json.dumps(brief_after_review_view, indent=2))

Persisted reviewer prompt:

Classify whether the brief explicitly supports the observed existing-row behavior.
Return SUPPORTED, CONTRADICTED, NOT_EVIDENCED, or UNCERTAIN. SUPPORTED and
CONTRADICTED require an exact quote from the brief; otherwise set evidence_quote
to an empty string.

Original brief:
Add 30-day expiration support to item-sharing links.

Store expiration in `public.share_links.expires_at` as a nullable timestamp with time
zone. New item-sharing links must expire 30 days after creation.


Observed rollout option: EXPIRE_EXISTING
Observed behavior: Existing item-sharing links receive an expiration approximately 30 days from migration; new links default to approximately 30 days after creation; expires_at remains nullable.

Validated reviewer result:

{
  "classification": "NOT_EVIDENCED",
  "evidence_quote": null
}

What exists after review:

{
  "authoritative_brief": "Add 30-day expiration support to item-sharing links.\n\nStore expiration in `public.share_links.expire

## 11. Inspect the durable pause and typed question

**Actor:** deterministic gate.

`NOT_EVIDENCED` or `UNCERTAIN` maps to `AWAITING_OWNER`. The fixed application vocabulary turns the observed policy into a narrow decision request with two verifiable choices. `decision_request` is derived for presentation; `run.json` persists the smaller `DecisionRecord`.


In [11]:
pause_summary = run_idg("show", RUN_ID)
print("\nPersisted decision record:\n")
print(json.dumps(START_SNAPSHOT["decision"], indent=2))
assert pause_summary["state"] == "AWAITING_OWNER"
assert pause_summary["decision_request"] is not None

$ uv run idg show 696f73f7991f46768159e7dcf2e05ad5
{
  "run_id": "696f73f7991f46768159e7dcf2e05ad5",
  "state": "AWAITING_OWNER",
  "observed_option": "EXPIRE_EXISTING",
  "classification": "NOT_EVIDENCED",
  "decision_request": {
    "id": "existing_item_sharing_link_rollout",
    "question": "What should happen to existing item-sharing links?",
    "reason": "The gate could not establish from the brief whether the 30-day expiration should apply to existing item-sharing links.",
    "observed": {
      "option": "EXPIRE_EXISTING",
      "behavior": "Existing item-sharing links receive an expiration approximately 30 days from migration; new links default to approximately 30 days after creation; expires_at remains nullable."
    },
    "options": [
      {
        "option": "PRESERVE_EXISTING",
        "behavior": "Existing item-sharing links remain non-expiring with NULL; new links default to approximately 30 days after creation; expires_at remains nullable.",
        "command": "uv ru

## 12. Complete the missing contract

**Actor:** human job owner.

Inspect the observed option and the two behaviors above, then review or edit the literal below. Either choice is valid: selecting the observed option explicitly confirms it, while selecting the other option makes a behavioral change easier to see in the SQL diff. The checked-in run records `EXPIRE_EXISTING`; the value is an explicit owner input and is not derived by a model or orchestration rule.


In [12]:
OBSERVED_OPTION = str(START_SNAPSHOT["decision"]["observed"])

# Human decision: edit this one value after reading the decision request.
OWNER_OPTION = "EXPIRE_EXISTING"

valid_owner_options = {"PRESERVE_EXISTING", "EXPIRE_EXISTING"}
if OWNER_OPTION not in valid_owner_options:
    raise ValueError(f"OWNER_OPTION must be one of {sorted(valid_owner_options)}")
policy_changed = OWNER_OPTION != OBSERVED_OPTION
print(f"Observed: {OBSERVED_OPTION}")
print(f"Owner selected: {OWNER_OPTION}")
print(f"Behavioral policy changed: {policy_changed}")
if not policy_changed:
    print("The owner confirmed the observed policy; this is a valid contract completion.")

Observed: EXPIRE_EXISTING
Owner selected: EXPIRE_EXISTING
Behavioral policy changed: False
The owner confirmed the observed policy; this is a valid contract completion.


**Actor:** human job owner, with the run store persisting the answer.

`answer` accepts only one of the two typed options. It records the decision and advances the durable state to `READY_TO_RESUME`; it does not call a model or interpret free text. This prototype trusts the local caller and does not authenticate or attribute the owner identity; a production integration must supply that control.


In [13]:
answer_summary = run_idg("answer", RUN_ID, "--option", OWNER_OPTION)
ANSWER_SNAPSHOT = copy.deepcopy(load_run(RUN_ID))
print("\nPersisted owner decision:\n")
print(json.dumps(ANSWER_SNAPSHOT["decision"], indent=2))
assert answer_summary["state"] == "READY_TO_RESUME"

$ uv run idg answer 696f73f7991f46768159e7dcf2e05ad5 --option EXPIRE_EXISTING
{
  "run_id": "696f73f7991f46768159e7dcf2e05ad5",
  "state": "READY_TO_RESUME",
  "observed_option": "EXPIRE_EXISTING",
  "classification": "NOT_EVIDENCED",
  "decision_request": null,
  "owner_option": "EXPIRE_EXISTING",
  "attempt_digests": [
    "78122b760223f63aeed3c26e56802fcc797136feac262d233196872bef232bf3"
  ],
  "final_worktree_path": null,
  "error": null
}

Persisted owner decision:

{
  "decision_id": "existing_item_sharing_link_rollout",
  "observed": "EXPIRE_EXISTING",
  "selected": "EXPIRE_EXISTING",
  "answered_at": "2026-08-23T15:00:24.796195Z"
}


## 13. Resume from durable state

**Actors:** gate orchestrator and a new coding-agent process.

`resume` can run later or in another process. It loads the recorded answer, creates a second clean worktree at the original commit, starts a fresh ephemeral Codex process, and probes the regenerated migration. The first model process is not resumed.


In [14]:
resume_summary = run_idg("resume", RUN_ID)
FINAL_SNAPSHOT = copy.deepcopy(load_run(RUN_ID))
attempt_two = FINAL_SNAPSHOT["attempts"][1]
assert len(FINAL_SNAPSHOT["attempts"]) == 2

$ uv run idg resume 696f73f7991f46768159e7dcf2e05ad5
{
  "run_id": "696f73f7991f46768159e7dcf2e05ad5",
  "state": "COMPLETED",
  "observed_option": "EXPIRE_EXISTING",
  "classification": "NOT_EVIDENCED",
  "decision_request": null,
  "owner_option": "EXPIRE_EXISTING",
  "attempt_digests": [
    "78122b760223f63aeed3c26e56802fcc797136feac262d233196872bef232bf3",
    "5849e3a3e69b23a78c13e8237bfcb5c1afd4b952ae613ccef3310144de960108"
  ],
  "final_worktree_path": "/home/user/projects/.implicit-decision-gate-idg-worktrees/696f73f7991f46768159e7dcf2e05ad5/attempt-2",
  "error": null
}


## 14. Inspect the fresh second request

**Actor:** gate prompt renderer supplying the new coding-agent process.

Attempt two receives the original brief and schema plus the authoritative owner choice, its required behavior, and PostgreSQL-specific acceptance criteria. It does not receive attempt one's SQL or the reviewer's rationale.


In [15]:
prompt_two = str(attempt_two["coding_prompt"])
print("Persisted attempt-two coding prompt:\n")
print(prompt_two)

isolation_checks = {
    "different_worktree": attempt_two["worktree_path"] != attempt_one["worktree_path"],
    "second_clean_start_verified": attempt_two["clean_start_verified"],
    "attempt_one_sql_absent": migration_one.strip() not in prompt_two,
    "reviewer_prompt_absent": START_SNAPSHOT["reviewer_prompt"] not in prompt_two,
    "owner_decision_present": f"Authoritative owner decision: {OWNER_OPTION}" in prompt_two,
}
print("\nContext isolation checks:\n")
print(json.dumps(isolation_checks, indent=2))
assert all(isolation_checks.values())

Persisted attempt-two coding prompt:

You create exactly one PostgreSQL migration.
Use only the supplied brief and baseline schema. Do not inspect or edit repository files.
Return the complete migration as structured SQL, without transaction-control statements.
The expires_at column must be a nullable timestamp with time zone. New rows must default
to approximately 30 days from creation.

Original brief:
Add 30-day expiration support to item-sharing links.

Store expiration in `public.share_links.expires_at` as a nullable timestamp with time
zone. New item-sharing links must expire 30 days after creation.


Baseline schema:
CREATE TABLE public.share_links (
    id bigint GENERATED ALWAYS AS IDENTITY PRIMARY KEY,
    token text NOT NULL UNIQUE,
    created_at timestamp with time zone NOT NULL DEFAULT CURRENT_TIMESTAMP
);

INSERT INTO public.share_links (token) VALUES ('existing-fixture');


Authoritative owner decision: EXPIRE_EXISTING
Required behavior: Existing item-sharing links rece

## 15. Compare the two proposed mechanisms

**Actors:** second coding agent for the new SQL and notebook operator acting as auditor.

The unified diff makes the agent's implementation change inspectable. It is not the correctness proof: that comes from the second PostgreSQL probe in the next stage. The metadata also demonstrates separate worktrees and immutable digests.


In [16]:
artifact_two_path = RUN_DIR / "attempt-2.sql"
migration_two = artifact_two_path.read_text(encoding="utf-8")
computed_digest_two = hashlib.sha256(migration_two.encode()).hexdigest()
migration_diff = "".join(
    difflib.unified_diff(
        migration_one.splitlines(keepends=True),
        migration_two.splitlines(keepends=True),
        fromfile="attempt-1.sql",
        tofile="attempt-2.sql",
    )
)

print("Attempt-two SQL artifact:\n")
print(migration_two)
print("Unified diff:\n")
print(migration_diff or "No textual difference between the two migrations.")
comparison = {
    "attempt_1": {
        "digest": attempt_one["migration_digest"],
        "worktree": attempt_one["worktree_path"],
    },
    "attempt_2": {
        "digest": attempt_two["migration_digest"],
        "computed_digest": computed_digest_two,
        "digest_matches": attempt_two["migration_digest"] == computed_digest_two,
        "worktree": attempt_two["worktree_path"],
    },
}
print(json.dumps(comparison, indent=2))
assert comparison["attempt_2"]["digest_matches"]

Attempt-two SQL artifact:

ALTER TABLE public.share_links
    ADD COLUMN expires_at timestamp with time zone;

UPDATE public.share_links
SET expires_at = CURRENT_TIMESTAMP + INTERVAL '30 days'
WHERE expires_at IS NULL;

ALTER TABLE public.share_links
    ALTER COLUMN expires_at SET DEFAULT (CURRENT_TIMESTAMP + INTERVAL '30 days');

Unified diff:

--- attempt-1.sql
+++ attempt-2.sql
@@ -1,3 +1,9 @@
 ALTER TABLE public.share_links
-    ADD COLUMN expires_at timestamp with time zone
-    DEFAULT (CURRENT_TIMESTAMP + INTERVAL '30 days');
+    ADD COLUMN expires_at timestamp with time zone;
+
+UPDATE public.share_links
+SET expires_at = CURRENT_TIMESTAMP + INTERVAL '30 days'
+WHERE expires_at IS NULL;
+
+ALTER TABLE public.share_links
+    ALTER COLUMN expires_at SET DEFAULT (CURRENT_TIMESTAMP + INTERVAL '30 days');

{
  "attempt_1": {
    "digest": "78122b760223f63aeed3c26e56802fcc797136feac262d233196872bef232bf3",
    "worktree": "/home/user/projects/.implicit-decision-gate-idg-worktrees/

## 16. Verify the completed contract

**Actors:** PostgreSQL verifier first, then deterministic gate.

PostgreSQL again reduces runtime behavior to the bounded `ProbeResult`. Final acceptance is the typed equality shown below: the observed rollout must equal the owner's selected rollout. No model judges whether attempt two succeeded.


In [17]:
probe_two = attempt_two["probe_result"]
selected_option = str(FINAL_SNAPSHOT["decision"]["selected"])
observed_option_two = str(probe_two["rollout_option"])
selected_matches_observed = selected_option == observed_option_two
final_verification = {
    "selected_by_owner": selected_option,
    "observed_by_postgresql": observed_option_two,
    "selected_equals_observed": selected_matches_observed,
    "final_state": FINAL_SNAPSHOT["state"],
    "error": FINAL_SNAPSHOT["error"],
    "probe_result": probe_two,
}
print(json.dumps(final_verification, indent=2))
assert selected_matches_observed
assert resume_summary["state"] == "COMPLETED"

{
  "selected_by_owner": "EXPIRE_EXISTING",
  "observed_by_postgresql": "EXPIRE_EXISTING",
  "selected_equals_observed": true,
  "final_state": "COMPLETED",
  "error": null,
  "probe_result": {
    "data_type": "timestamp with time zone",
    "nullable": true,
    "column_default": "(CURRENT_TIMESTAMP + '30 days'::interval)",
    "insert_without_value": "approximately_now_plus_30_days",
    "existing_row": "approximately_migration_time_plus_30_days",
    "rollout_option": "EXPIRE_EXISTING",
    "rollback_verified": true
  }
}


## 17. Inspect the durable record

**Actor:** auditor reading the run store.

The complete `run.json` below is the atomically replaced current-state snapshot. It is not an append-only event log or a hidden graph. Its ordered attempts, decision timestamps, prompts, probe results, and adjacent immutable SQL files are the inspectable state of this run.


In [18]:
run_json_path = RUN_DIR / "run.json"
print("Complete run.json:\n")
print(run_json_path.read_text(encoding="utf-8"))

timeline = [
    {"event": "run_created", "at": FINAL_SNAPSHOT["created_at"]},
    {
        "event": "attempt_1_completed",
        "at": FINAL_SNAPSHOT["attempts"][0]["completed_at"],
    },
    {"event": "owner_answered", "at": FINAL_SNAPSHOT["decision"]["answered_at"]},
    {
        "event": "attempt_2_completed",
        "at": FINAL_SNAPSHOT["attempts"][1]["completed_at"],
    },
    {"event": "snapshot_updated", "at": FINAL_SNAPSHOT["updated_at"]},
]
artifacts = [
    {"name": path.name, "bytes": path.stat().st_size}
    for path in sorted(RUN_DIR.glob("attempt-*.sql"))
]
print("Derived timeline:\n")
print(json.dumps(timeline, indent=2))
print("\nImmutable SQL artifacts:\n")
print(json.dumps(artifacts, indent=2))

Complete run.json:

{
  "run_id": "696f73f7991f46768159e7dcf2e05ad5",
  "state": "COMPLETED",
  "original_brief": "Add 30-day expiration support to item-sharing links.\n\nStore expiration in `public.share_links.expires_at` as a nullable timestamp with time\nzone. New item-sharing links must expire 30 days after creation.\n",
  "base_commit": "7703bce2b4afc78a98765346decf7953b032d91b",
  "attempts": [
    {
      "number": 1,
      "worktree_path": "/home/user/projects/.implicit-decision-gate-idg-worktrees/696f73f7991f46768159e7dcf2e05ad5/attempt-1",
      "clean_start_verified": true,
      "coding_prompt": "You create exactly one PostgreSQL migration.\nUse only the supplied brief and baseline schema. Do not inspect or edit repository files.\nReturn the complete migration as structured SQL, without transaction-control statements.\nThe expires_at column must be a nullable timestamp with time zone. New rows must default\nto approximately 30 days from creation.\n\nOriginal brief:\nAdd 30-

## 18. Current boundary and possible extension

**Actor:** system designer.

What exists in this vertical slice:

- A commit-pinned authoritative brief and schema.
- A coding agent constrained to structured SQL output.
- A real PostgreSQL observation normalized into one typed policy vocabulary.
- A narrow AI evidence review of one observed hypothesis.
- A durable typed human amendment and clean-context retry.
- A deterministic final comparison between selected and observed behavior.

What does not exist yet:

- A general semantic compiler that turns arbitrary prose into a trusted typed contract.
- Authenticated and attributable agent or human identity.
- Persisted raw database observations or raw model transcripts.
- An append-only event ledger or graph database.

A future semantic verifier could let AI propose typed claims, unknown fields, and source spans for human approval, then compare the approved policy with normalized runtime evidence. The AI-derived interpretation must not silently replace the human-owned brief. The current reviewer is also deliberately weaker than that vision: literal quote validation proves source occurrence, not semantic relevance.


## 19. Optional infrastructure cleanup

**Actor:** notebook operator.

Stopping Compose removes the local verifier container after the walkthrough. It does not remove `.idg/runs/<run_id>`, the SQL artifacts, or the detached worktrees, so the evidence remains available for manual inspection. Set the flag to `True` when you are finished.


In [19]:
STOP_DOCKER = False

if EXTERNAL_POSTGRES:
    print("No Compose-managed PostgreSQL instance was started.")
elif STOP_DOCKER:
    run_command(["docker", "compose", "down"])
else:
    print("PostgreSQL is still running. Set STOP_DOCKER = True and rerun this cell to stop it.")

PostgreSQL is still running. Set STOP_DOCKER = True and rerun this cell to stop it.
